In [18]:
# !mkdir -p /data/tlc_hvfhv_2024
# !for m in $(seq -w 1 12); do \
#   url="https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2024-${m}.parquet"; \
#   out="/data/tlc_hvfhv_2024/fhvhv_tripdata_2024-${m}.parquet"; \
#   echo "Downloading 2024-${m} -> $out"; \
#   curl -L -A "Mozilla/5.0" --retry 8 --retry-all-errors --retry-delay 2 -o "$out" "$url" || exit 1; \
# done


| 字段名                  | 中文说明                                               |
| -------------------- | -------------------------------------------------- |
| hvfhs_license_num    | HVFHS 平台/公司的 TLC 牌照号（如 HV0003=Uber、HV0005=Lyft 等）。 |
| dispatching_base_num | 派单/调度该行程的 Base（车队/基地）许可证号。                         |
| originating_base_num | 最初接收乘客请求的 Base 号（你的数据中约 25% 缺失）。                   |
| request_datetime     | 乘客发起叫车请求的时间。                                       |
| on_scene_datetime    | 司机到达上车点时间（通常仅无障碍/特定车辆记录；你的数据中约 25% 缺失）。            |
| pickup_datetime      | 实际上车时间。                                            |
| dropoff_datetime     | 实际下车时间。                                            |
| PULocationID         | 上车区域（TLC Taxi Zone）编号。                             |
| DOLocationID         | 下车区域（TLC Taxi Zone）编号。                             |
| trip_miles           | 该订单行程里程（英里）。                                       |
| trip_time            | 该订单行程时长（秒）。                                        |
| base_passenger_fare  | 基础车费（不含过路费/小费/税费/附加费）。                             |
| tolls                | 过路费总额。                                             |
| bcf                  | Black Car Fund 收费总额。                               |
| sales_tax            | 纽约州销售税总额。                                          |
| congestion_surcharge | 拥堵费总额。                                             |
| airport_fee          | 机场附加费（JFK/LGA/EWR 等机场上下客通常固定收费）。                   |
| tips                 | 乘客支付的小费总额。                                         |
| driver_pay           | 司机收入（不含过路费/小费，且已扣除抽佣/附加费/税费等）。                     |
| shared_request_flag  | 乘客是否选择拼车（不论是否最终匹配）（Y/N）。                           |
| shared_match_flag    | 行程过程中是否实际发生拼车匹配（Y/N）。                              |
| access_a_ride_flag   | 是否为 MTA “Access-a-Ride” 项目行程（Y/N）。                 |
| wav_request_flag     | 乘客是否请求无障碍车辆（WAV）（Y/N）。                             |
| wav_match_flag       | 行程是否实际由无障碍车辆（WAV）完成（Y/N）。                          |


In [10]:

import os
import polars as pl

DATA_DIR = "/data/tlc_hvfhv_2024"
IN_TMPL  = "fhvhv_tripdata_2024-{m:02d}.parquet"
OUT_DIR  = "/data/tlc_hvfhv_2024_out"
os.makedirs(OUT_DIR, exist_ok=True)

def collect_streaming(lf: pl.LazyFrame) -> pl.DataFrame:
    try:
        return lf.collect(engine="streaming")
    except TypeError:
        return lf.collect()

def build_month_overview_row(m: int) -> pl.DataFrame:
    path = os.path.join(DATA_DIR, IN_TMPL.format(m=m))

    use_cols = [
        "request_datetime",
        "PULocationID", "DOLocationID",
        "trip_miles", "trip_time",
        "hvfhs_license_num",
        "dispatching_base_num",
        "originating_base_num",
        "shared_request_flag", "wav_request_flag", "access_a_ride_flag",
        "base_passenger_fare", "tolls", "bcf", "sales_tax", "congestion_surcharge", "airport_fee",
        "tips",
    ]

    lf = pl.scan_parquet(path).select(use_cols)

    # y_cost_no_tips（缺失当0）
    y_cost = (
        pl.col("base_passenger_fare").fill_null(0) +
        pl.col("tolls").fill_null(0) +
        pl.col("bcf").fill_null(0) +
        pl.col("sales_tax").fill_null(0) +
        pl.col("congestion_surcharge").fill_null(0) +
        pl.col("airport_fee").fill_null(0)
    ).alias("y_cost_no_tips")

    lf = lf.with_columns(y_cost)

    # --- 主聚合（全样本）---
    agg_main = lf.select([
        pl.len().alias("n_rows"),
        pl.col("request_datetime").min().alias("min_request_dt"),
        pl.col("request_datetime").max().alias("max_request_dt"),

        # null counts
        pl.col("originating_base_num").null_count().alias("originating_base_nulls"),
        pl.col("dispatching_base_num").null_count().alias("dispatching_base_nulls"),
        pl.col("hvfhs_license_num").null_count().alias("hvfhs_nulls"),
        pl.col("PULocationID").null_count().alias("PU_nulls"),
        pl.col("DOLocationID").null_count().alias("DO_nulls"),
        pl.col("trip_miles").null_count().alias("trip_miles_nulls"),
        pl.col("trip_time").null_count().alias("trip_time_nulls"),
        pl.col("y_cost_no_tips").null_count().alias("cost_nulls"),
        pl.col("tips").null_count().alias("tips_nulls"),

        # trip_miles summary
        pl.col("trip_miles").mean().alias("trip_miles_mean"),
        pl.col("trip_miles").std().alias("trip_miles_std"),
        pl.col("trip_miles").min().alias("trip_miles_min"),
        pl.col("trip_miles").max().alias("trip_miles_max"),

        # 0-miles count（只在主表里统计，避免重复列名）
        (pl.col("trip_miles") == 0).sum().alias("trip_miles_zero_cnt"),

        # trip_time summary
        pl.col("trip_time").mean().alias("trip_time_mean"),
        pl.col("trip_time").std().alias("trip_time_std"),
        pl.col("trip_time").min().alias("trip_time_min"),
        pl.col("trip_time").max().alias("trip_time_max"),

        # cost summary
        pl.col("y_cost_no_tips").mean().alias("cost_mean"),
        pl.col("y_cost_no_tips").std().alias("cost_std"),
        pl.col("y_cost_no_tips").min().alias("cost_min"),
        pl.col("y_cost_no_tips").max().alias("cost_max"),

        # tips summary
        pl.col("tips").mean().alias("tips_mean"),
        pl.col("tips").min().alias("tips_min"),
        pl.col("tips").max().alias("tips_max"),
   
         # --- ✅ categorical cardinality (n_unique) ---
        pl.col("PULocationID").n_unique().alias("PU_n_unique"),
        pl.col("DOLocationID").n_unique().alias("DO_n_unique"),

        # 这三个是字符串类：先统一 cast + fill_null 再 n_unique
        pl.col("hvfhs_license_num").cast(pl.Utf8).fill_null("UNK").n_unique().alias("hvfhs_n_unique"),
        pl.col("dispatching_base_num").cast(pl.Utf8).fill_null("UNK").n_unique().alias("dispatch_n_unique"),
        pl.col("originating_base_num").cast(pl.Utf8).fill_null("UNK").n_unique().alias("origin_n_unique"),

        # route_id: PU*1000 + DO
        (pl.col("PULocationID").cast(pl.Int32) * 1000 + pl.col("DOLocationID").cast(pl.Int32))
            .n_unique()
            .alias("route_n_unique"),
    ])
    # --- 0里程子样本画像（不要再输出 trip_miles_zero_cnt）---
    agg_zero = (
        lf.filter(pl.col("trip_miles") == 0)
          .select([
              pl.len().alias("miles0_rows"),
              pl.col("trip_time").mean().alias("trip_time_mean_when_miles0"),
              pl.col("y_cost_no_tips").mean().alias("cost_mean_when_miles0"),
              pl.col("tips").mean().alias("tips_mean_when_miles0"),
          ])
    )

    df_main = collect_streaming(agg_main)
    df_zero = collect_streaming(agg_zero)

    # 横向合并（列名不冲突）
    df = pl.concat([df_main, df_zero], how="horizontal").with_columns(
        pl.lit(f"2024-{m:02d}").alias("month")
    )

    # 0里程占比
    df = df.with_columns(
        (pl.col("trip_miles_zero_cnt") / pl.col("n_rows")).alias("trip_miles_zero_rate")
    )

    # 统一时间精度（避免 concat schema error）
    df = df.with_columns([
        pl.col("min_request_dt").cast(pl.Datetime("ns")),
        pl.col("max_request_dt").cast(pl.Datetime("ns")),
    ])

    # 缺失率
    df = df.with_columns([
        (pl.col("originating_base_nulls") / pl.col("n_rows")).alias("originating_base_null_rate"),
        (pl.col("dispatching_base_nulls") / pl.col("n_rows")).alias("dispatching_base_null_rate"),
        (pl.col("hvfhs_nulls") / pl.col("n_rows")).alias("hvfhs_null_rate"),
        (pl.col("PU_nulls") / pl.col("n_rows")).alias("PU_null_rate"),
        (pl.col("DO_nulls") / pl.col("n_rows")).alias("DO_null_rate"),
        (pl.col("trip_miles_nulls") / pl.col("n_rows")).alias("trip_miles_null_rate"),
        (pl.col("trip_time_nulls") / pl.col("n_rows")).alias("trip_time_null_rate"),
        (pl.col("cost_nulls") / pl.col("n_rows")).alias("cost_null_rate"),
        (pl.col("tips_nulls") / pl.col("n_rows")).alias("tips_null_rate"),
    ])

    # cast float32（省空间）
    float_cols = [
        "trip_miles_mean","trip_miles_std","trip_miles_min","trip_miles_max",
        "trip_time_mean","trip_time_std","trip_time_min","trip_time_max",
        "cost_mean","cost_std","cost_min","cost_max",
        "tips_mean","tips_min","tips_max",
        "originating_base_null_rate","dispatching_base_null_rate","hvfhs_null_rate",
        "PU_null_rate","DO_null_rate","trip_miles_null_rate","trip_time_null_rate",
        "cost_null_rate","tips_null_rate",
        "trip_time_mean_when_miles0","cost_mean_when_miles0","tips_mean_when_miles0",
        "trip_miles_zero_rate",
    ]
    df = df.with_columns([pl.col(c).cast(pl.Float32) for c in float_cols if c in df.columns])

    return df


# --------- 跑全年并写盘 ----------
rows = []
for m in range(1, 13):
    rows.append(build_month_overview_row(m))
    print(f"[OK] overview month={m:02d}")


[OK] overview month=01
[OK] overview month=02
[OK] overview month=03
[OK] overview month=04
[OK] overview month=05
[OK] overview month=06
[OK] overview month=07
[OK] overview month=08
[OK] overview month=09
[OK] overview month=10
[OK] overview month=11
[OK] overview month=12


In [12]:
import os
import polars as pl

DATA_DIR = "/data/tlc_hvfhv_2024"
IN_TMPL  = "fhvhv_tripdata_2024-{m:02d}.parquet"
OUT_DIR  = "/data/tlc_hvfhv_2024_out"
os.makedirs(OUT_DIR, exist_ok=True)

TARGET_Q_PATH_PARQUET = os.path.join(OUT_DIR, "target_quantiles_2024.parquet")
TARGET_Q_PATH_CSV     = os.path.join(OUT_DIR, "target_quantiles_2024.csv")

def collect_streaming(lf: pl.LazyFrame) -> pl.DataFrame:
    try:
        return lf.collect(engine="streaming")
    except TypeError:
        return lf.collect()

def target_quantiles_2024() -> pl.DataFrame:
    """
    计算全年目标变量分位数与最大值：p50/p90/p99/p99.5/max
    - y_cost_no_tips: 由费用项求和（null当0）
    - y_time: trip_time（秒）
    输出两行，统一为 Float64，避免 concat schema 冲突
    """
    paths = [os.path.join(DATA_DIR, IN_TMPL.format(m=m)) for m in range(1, 13)]

    y_cost = (
        pl.col("base_passenger_fare").fill_null(0) +
        pl.col("tolls").fill_null(0) +
        pl.col("bcf").fill_null(0) +
        pl.col("sales_tax").fill_null(0) +
        pl.col("congestion_surcharge").fill_null(0) +
        pl.col("airport_fee").fill_null(0)
    ).cast(pl.Float64).alias("y_cost_no_tips")

    # 时间也先 cast 成 Float64，确保 quantile/max 都是 Float64
    y_time = pl.col("trip_time").cast(pl.Float64).alias("y_time")

    lf = (
        pl.scan_parquet(paths)
          .select([y_cost, y_time])
          .filter((pl.col("y_cost_no_tips") >= 0) & (pl.col("y_time") > 0))
    )

    def q_row(col: str, name: str) -> pl.LazyFrame:
        return (
            lf.select([
                pl.col(col).quantile(0.50, "nearest").cast(pl.Float64).alias("p50"),
                pl.col(col).quantile(0.90, "nearest").cast(pl.Float64).alias("p90"),
                pl.col(col).quantile(0.99, "nearest").cast(pl.Float64).alias("p99"),
                pl.col(col).quantile(0.995, "nearest").cast(pl.Float64).alias("p99_5"),
                pl.col(col).max().cast(pl.Float64).alias("max"),
            ])
            .with_columns(pl.lit(name).alias("target"))
            .select(["target", "p50", "p90", "p99", "p99_5", "max"])
        )

    out = pl.concat(
        [q_row("y_cost_no_tips", "y_cost_no_tips"), q_row("y_time", "y_time")],
        how="vertical"
    )

    return collect_streaming(out)

# ---- 计算 + 保存 parquet + csv ----
qdf = target_quantiles_2024()
qdf.write_parquet(TARGET_Q_PATH_PARQUET)
qdf.write_csv(TARGET_Q_PATH_CSV)

print("[SAVED]", TARGET_Q_PATH_PARQUET)
print("[SAVED]", TARGET_Q_PATH_CSV)
print(qdf)


[SAVED] /data/tlc_hvfhv_2024_out/target_quantiles_2024.parquet
[SAVED] /data/tlc_hvfhv_2024_out/target_quantiles_2024.csv
shape: (2, 6)
┌────────────────┬───────┬────────┬────────┬────────┬─────────┐
│ target         ┆ p50   ┆ p90    ┆ p99    ┆ p99_5  ┆ max     │
│ ---            ┆ ---   ┆ ---    ┆ ---    ┆ ---    ┆ ---     │
│ str            ┆ f64   ┆ f64    ┆ f64    ┆ f64    ┆ f64     │
╞════════════════╪═══════╪════════╪════════╪════════╪═════════╡
│ y_cost_no_tips ┆ 22.95 ┆ 61.96  ┆ 142.03 ┆ 170.92 ┆ 2578.31 │
│ y_time         ┆ 978.0 ┆ 2308.0 ┆ 4293.0 ┆ 4914.0 ┆ 55138.0 │
└────────────────┴───────┴────────┴────────┴────────┴─────────┘


In [ ]:
# import pandas as pd
# import holidays

# OUT_PATH = "/data/tlc_hvfhv_2024_out/holidays_NY_2024.parquet"

# hcal = holidays.US(subdiv="NY", years=[2024])  # NY 州假期 :contentReference[oaicite:2]{index=2}

# dates = pd.date_range("2024-01-01", "2024-12-31", freq="D")
# dfh = pd.DataFrame({"date": dates.date})
# dfh["is_holiday"] = [d in hcal for d in dfh["date"]]
# dfh["holiday_name"] = [hcal.get(d, "") for d in dfh["date"]]

# # 前后一天（更有用）
# is_h = dfh["is_holiday"].to_numpy()
# dfh["is_before_holiday"] = pd.Series(is_h).shift(-1, fill_value=False).to_numpy()
# dfh["is_after_holiday"]  = pd.Series(is_h).shift(+1, fill_value=False).to_numpy()

# dfh.to_parquet(OUT_PATH, index=False)
# print("saved:", OUT_PATH, "rows:", len(dfh))
# print(dfh[dfh["is_holiday"]].head())
# import pandas as pd
# import requests

# OUT_PATH = "/data/tlc_hvfhv_2024_out/weather_hourly_2024.parquet"

# def fetch_open_meteo_hourly_2024(
#     lat=40.7128, lon=-74.0060, timezone="America/New_York"
# ):
#     url = "https://archive-api.open-meteo.com/v1/archive"
#     params = {
#         "latitude": lat,
#         "longitude": lon,
#         "start_date": "2024-01-01",
#         "end_date": "2024-12-31",
#         "hourly": ",".join([
#             "temperature_2m",
#             "precipitation",
#             "wind_speed_10m",
#             "weather_code",
#         ]),
#         "timezone": timezone,
#     }

#     r = requests.get(url, params=params, timeout=120)
#     r.raise_for_status()
#     js = r.json()

#     df = pd.DataFrame({"ts_hour": pd.to_datetime(js["hourly"]["time"])})
#     for k, v in js["hourly"].items():
#         if k != "time":
#             df[k] = v

#     # 简单清理
#     df = df.sort_values("ts_hour").reset_index(drop=True)
#     return df

# dfw = fetch_open_meteo_hourly_2024()
# dfw.to_parquet(OUT_PATH, index=False)
# print("saved:", OUT_PATH, "rows:", len(dfw))
# print(dfw.head())


In [8]:
print("[SAVED]", overview_path)
print(overview_2024)
overview_2024 = pl.concat(rows, how="vertical")
overview_path = os.path.join(OUT_DIR, "overview_2024.parquet")
overview_2024.write_parquet(overview_path)

[SAVED] /data/tlc_hvfhv_2024_out/overview_2024.parquet
shape: (12, 43)
┌──────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ n_rows   ┆ min_reque ┆ max_reque ┆ originati ┆ … ┆ trip_mile ┆ trip_time ┆ cost_null ┆ tips_null │
│ ---      ┆ st_dt     ┆ st_dt     ┆ ng_base_n ┆   ┆ s_null_ra ┆ _null_rat ┆ _rate     ┆ _rate     │
│ u32      ┆ ---       ┆ ---       ┆ ulls      ┆   ┆ te        ┆ e         ┆ ---       ┆ ---       │
│          ┆ datetime[ ┆ datetime[ ┆ ---       ┆   ┆ ---       ┆ ---       ┆ f32       ┆ f32       │
│          ┆ ns]       ┆ ns]       ┆ u32       ┆   ┆ f32       ┆ f32       ┆           ┆           │
╞══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 19663930 ┆ 2023-12-3 ┆ 2024-02-0 ┆ 5218737   ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0       │
│          ┆ 1         ┆ 1         ┆           ┆   ┆           ┆           ┆           ┆           │
│          ┆ 23:01:5

In [9]:
overview_2024 = pl.concat(rows, how="vertical")

overview_path = os.path.join(OUT_DIR, "overview_2024.parquet")
overview_2024.write_parquet(overview_path)

overview_csv_path = os.path.join(OUT_DIR, "overview_2024.csv")
overview_2024.write_csv(overview_csv_path)

print("[SAVED]", overview_path)
print("[SAVED]", overview_csv_path)


[SAVED] /data/tlc_hvfhv_2024_out/overview_2024.parquet
[SAVED] /data/tlc_hvfhv_2024_out/overview_2024.csv


我们以 **request_datetime（乘客发起请求时刻）** 为预测时点进行建模，避免使用在行程开始/结束后才能观测到的信息（如 `pickup_datetime`、`dropoff_datetime`、`on_scene_datetime` 等），以防止信息泄露。预测目标包含两部分：① **乘客开销（不含 tips）**，由数据字典中的费用明细项构造
[
y_{\text{cost}} = \text{base_passenger_fare}+\text{tolls}+\text{bcf}+\text{sales_tax}+\text{congestion_surcharge}+\text{airport_fee}
]
其中各费用项的缺失值按 0 处理（代表未收取/不适用）；② **乘坐时间**，直接使用 `trip_time`（秒）作为标签。

特征工程遵循“简单、可解释、计算开销小”的原则，主要包括四类特征：
(1) **时间特征**：由 `request_datetime` 提取 `hour`（小时）、`dow`（星期几）、`is_weekend`（是否周末）、`month`（月份）以及 `is_peak`（早晚高峰指示变量），以刻画需求与拥堵的周期性变化。
(2) **空间/路线特征**：使用 `PULocationID` 与 `DOLocationID` 表示上下车区域，并构造 `same_zone`（起终点是否同区）与 `route_id = PULocationID×1000 + DOLocationID` 作为“OD 路线”类别特征，用于捕捉不同出行走廊在价格与耗时上的系统差异。
(3) **行程长度特征**：加入 `trip_miles` 以及其非线性变换 `log1p_trip_miles`，基于“网约车下单时路线与里程通常已确定/可由平台估计”的假设，用以增强对费用与时间的解释能力并提升模型稳定性。
(4) **平台与服务属性特征**：将 `hvfhs_license_num`、`dispatching_base_num`、`originating_base_num` 等平台/基地字段做全局一致的标签编码；对 `shared_request_flag`、`wav_request_flag`、`access_a_ride_flag` 等请求时可知的服务属性标志进行二值化（Y/N→0/1）。

针对缺失值，除费用项按 0 填充外，`originating_base_num` 存在约 25% 缺失：我们将其缺失填为特殊类别“UNK”，并额外构造缺失指示变量 `originating_base_missing`（1 表示缺失）以保留缺失本身的可解释信息；`on_scene_datetime` 虽也存在较高缺失，但其语义属于司机到达信息，不满足 request 时点可用性，因此不纳入特征集合。最后，对明显异常样本进行过滤（如 `trip_time<=0`、`trip_miles<0`、`y_cost<0`），并将生成的特征按月份写出为独立的 features parquet 文件，以支持后续按时间切分训练/验证/测试并保证内存安全。


| 字段名                      | 类型/取值范围    | 中文说明（构造方式）                                                                                                      |
| ------------------------ | ---------- | --------------------------------------------------------------------------------------------------------------- |
| y_cost_no_tips           | float32    | **预测目标：不含小费的乘客开销**，由 `base_passenger_fare + tolls + bcf + sales_tax + congestion_surcharge + airport_fee` 相加得到。 |
| y_time                   | int32      | **预测目标：行程时长（秒）**，等于源字段 `trip_time`。                                                                             |
| hour                     | int8（0–23） | 请求时间的小时（从 `request_datetime` 提取）。                                                                               |
| dow                      | int8（0–6）  | 请求时间的星期（从 `request_datetime` 提取，0=周一/或按你实现定义）。                                                                  |
| is_weekend               | int8（0/1）  | 是否周末（由 `request_datetime` 判断）。                                                                                  |
| month                    | int8（1–12） | 请求时间的月份（由 `request_datetime` 提取）。                                                                               |
| is_peak                  | int8（0/1）  | 是否高峰时段（如 7–10 或 16–19 点，按你规则定义）。                                                                                |
| PULocationID             | int16      | 上车区域 Taxi Zone 编号。                                                                                              |
| DOLocationID             | int16      | 下车区域 Taxi Zone 编号。                                                                                              |
| same_zone                | int8（0/1）  | 上下车是否同一区域（`PULocationID == DOLocationID`）。                                                                      |
| route_id                 | int32      | OD 路线编码：`PULocationID * 1000 + DOLocationID`（用于表示一条具体上下车组合）。                                                    |
| trip_miles               | float32    | 行程里程（英里）（你假设下单时可由导航/计价估算得到）。                                                                                    |
| log1p_trip_miles         | float32    | `log(1 + trip_miles)`，用于缓解里程的长尾分布。                                                                              |
| shared_request_flag      | int8（0/1）  | 是否选择拼车（Y→1，否则 0）。                                                                                               |
| wav_request_flag         | int8（0/1）  | 是否请求无障碍车（Y→1，否则 0）。                                                                                             |
| access_a_ride_flag       | int8（0/1）  | 是否为 Access-a-Ride 行程（Y→1，否则 0）。                                                                                 |
| originating_base_missing | int8（0/1）  | `originating_base_num` 是否缺失（缺失=1，否则 0）。                                                                         |
| hvfhs_code               | int16      | 平台牌照号 `hvfhs_license_num` 的整数编码（UNK/缺失映射为 0）。                                                                   |
| dispatch_code            | int16      | 调度基地号 `dispatching_base_num` 的整数编码（UNK/缺失映射为 0）。                                                                |
| origin_code              | int16      | 发起基地号 `originating_base_num` 的整数编码（UNK/缺失映射为 0）。                                                                |
| temperature_2m           | float32    | 天气特征：NYC 单点的**逐小时气温**（按 `request_datetime` 截断到小时后 join）。                                                        |
| precipitation            | float32    | 天气特征：NYC 单点的**逐小时降水量**（按小时 join）。                                                                               |
| wind_speed_10m           | float32    | 天气特征：NYC 单点的**逐小时风速**（按小时 join）。                                                                                |
| weather_code             | int16      | 天气现象编码（open-meteo 的 weather code；缺失填 -1）。                                                                       |
| is_precip                | int8（0/1）  | 是否发生降水（`precipitation > 0` 记为 1）。                                                                               |
| weather_missing          | int8（0/1）  | 天气 join 是否缺失（缺失=1，否则 0）。                                                                                        |
| is_holiday               | int8（0/1）  | 是否纽约州法定/公共假期（按日期 join）。                                                                                         |
| is_before_holiday        | int8（0/1）  | 是否“假期前一天”（次日为假期）。                                                                                               |
| is_after_holiday         | int8（0/1）  | 是否“假期后一天”（前一日为假期）。                                                                                              |


In [2]:
# import os
# import polars as pl


import os
import polars as pl

# ---------- Paths ----------
DATA_DIR = "/data/tlc_hvfhv_2024"
IN_TMPL = "fhvhv_tripdata_2024-{m:02d}.parquet"
DATA_GLOB = f"{DATA_DIR}/fhvhv_tripdata_2024-*.parquet"

FEATURE_DIR_OUT = "/data/tlc_hvfhv_2024_features_wxhol"
os.makedirs(FEATURE_DIR_OUT, exist_ok=True)

WEATHER_PATH = "/data/tlc_hvfhv_2024_out/weather_hourly_2024.parquet"
HOLIDAY_PATH = "/data/tlc_hvfhv_2024_out/holidays_NY_2024.parquet"


# ---------- Utils ----------
def collect_streaming(lf: pl.LazyFrame) -> pl.DataFrame:
    """Compat: polars streaming collect."""
    try:
        return lf.collect(engine="streaming")
    except TypeError:
        return lf.collect()


def yn01(colname: str) -> pl.Expr:
    """'Y'/'N'/null -> 1/0 (null->0) Int8"""
    return (
        pl.when(pl.col(colname) == "Y").then(1).otherwise(0)
        .cast(pl.Int8)
        .alias(colname)
    )


# ---------- 1) Build global label-encoding maps (UNK=0) ----------
def _unique_values_one_col(col: str) -> list[str]:
    # scan only one column -> unique -> collect small list
    lf = pl.scan_parquet(DATA_GLOB).select(pl.col(col).cast(pl.Utf8).fill_null("UNK").alias(col))
    df = collect_streaming(lf.unique())
    vals = df.get_column(col).to_list()
    # ensure "UNK" exists
    if "UNK" not in vals:
        vals.append("UNK")
    return vals


def _make_map(vals: list[str]) -> dict[str, int]:
    vals = [v for v in vals if v != "UNK"]
    vals.sort()
    mp = {"UNK": 0}
    for i, v in enumerate(vals, start=1):
        mp[v] = i
    return mp


def build_global_maps():
    hv_vals = _unique_values_one_col("hvfhs_license_num")
    disp_vals = _unique_values_one_col("dispatching_base_num")
    orig_vals = _unique_values_one_col("originating_base_num")

    hv_map = _make_map(hv_vals)
    disp_map = _make_map(disp_vals)
    orig_map = _make_map(orig_vals)

    print(f"[maps] hvfhs={len(hv_map)} dispatch={len(disp_map)} originating={len(orig_map)}")
    return hv_map, disp_map, orig_map


# ---------- 2) Dimension tables (lazy) ----------
# Weather: one row per hour
weather_lf = (
    pl.scan_parquet(WEATHER_PATH)
    .with_columns([
        pl.col("ts_hour").cast(pl.Datetime("us")),
        pl.col("temperature_2m").cast(pl.Float32),
        pl.col("precipitation").cast(pl.Float32),
        pl.col("wind_speed_10m").cast(pl.Float32),
        pl.col("weather_code").cast(pl.Int32),
    ])
    .select(["ts_hour", "temperature_2m", "precipitation", "wind_speed_10m", "weather_code"])
)

# Holiday: IMPORTANT -> cast bool/int to Int8 here
holiday_lf = (
    pl.scan_parquet(HOLIDAY_PATH)
    .with_columns([
        pl.col("date").cast(pl.Date),

        # 关键：统一成 Int8，彻底避免 bool/int 混用导致 fill_null 报错
        pl.col("is_holiday").cast(pl.Int8),
        pl.col("is_before_holiday").cast(pl.Int8),
        pl.col("is_after_holiday").cast(pl.Int8),
    ])
    .select(["date", "is_holiday", "is_before_holiday", "is_after_holiday"])
)


# ---------- 3) Build one month features ----------
def build_month_features_wxhol(m: int, hv_map: dict, disp_map: dict, orig_map: dict) -> str:
    in_path = os.path.join(DATA_DIR, IN_TMPL.format(m=m))
    out_path = os.path.join(FEATURE_DIR_OUT, f"features_2024-{m:02d}.parquet")

    use_cols = [
        "request_datetime",
        "PULocationID", "DOLocationID",
        "trip_miles", "trip_time",
        "hvfhs_license_num",
        "dispatching_base_num",
        "originating_base_num",
        "shared_request_flag", "wav_request_flag", "access_a_ride_flag",
        "base_passenger_fare", "tolls", "bcf", "sales_tax", "congestion_surcharge", "airport_fee",
    ]

    lf = pl.scan_parquet(in_path).select(use_cols)

    # --- targets ---
    y_cost = (
        pl.col("base_passenger_fare").fill_null(0.0) +
        pl.col("tolls").fill_null(0.0) +
        pl.col("bcf").fill_null(0.0) +
        pl.col("sales_tax").fill_null(0.0) +
        pl.col("congestion_surcharge").fill_null(0.0) +
        pl.col("airport_fee").fill_null(0.0)
    ).cast(pl.Float32).alias("y_cost_no_tips")

    # --- normalize time dtype ---
    req_dt = pl.col("request_datetime").cast(pl.Datetime("us")).alias("request_datetime")

    # --- feature block ---
    feats = (
        lf
        .with_columns([
            req_dt,

            y_cost,
            pl.col("trip_time").cast(pl.Int32).alias("y_time"),

            # time features
            pl.col("request_datetime").dt.hour().cast(pl.Int8).alias("hour"),
            (pl.col("request_datetime").dt.weekday() - 1).cast(pl.Int8).alias("dow"),  # 0-6
            (pl.col("request_datetime").dt.weekday().is_in([6, 7])).cast(pl.Int8).alias("is_weekend"),
            pl.col("request_datetime").dt.month().cast(pl.Int8).alias("month"),
            (
                pl.col("request_datetime").dt.hour().is_between(7, 10) |
                pl.col("request_datetime").dt.hour().is_between(16, 19)
            ).cast(pl.Int8).alias("is_peak"),

            # geo/route
            pl.col("PULocationID").cast(pl.Int16).alias("PULocationID"),
            pl.col("DOLocationID").cast(pl.Int16).alias("DOLocationID"),
            (pl.col("PULocationID") == pl.col("DOLocationID")).cast(pl.Int8).alias("same_zone"),
            (pl.col("PULocationID").cast(pl.Int32) * 1000 + pl.col("DOLocationID").cast(pl.Int32)).cast(pl.Int32).alias("route_id"),

            # miles
            pl.col("trip_miles").cast(pl.Float32).alias("trip_miles"),
            pl.col("trip_miles").cast(pl.Float32).log1p().alias("log1p_trip_miles"),
            (pl.col("trip_miles").cast(pl.Float32) == 0.0).cast(pl.Int8).alias("is_zero_miles"),

            # flags
            yn01("shared_request_flag"),
            yn01("wav_request_flag"),
            yn01("access_a_ride_flag"),

            # missing indicator
            pl.col("originating_base_num").is_null().cast(pl.Int8).alias("originating_base_missing"),

            # global encoding (replace_strict to avoid deprecation)
            pl.col("hvfhs_license_num")
              .cast(pl.Utf8).fill_null("UNK")
              .replace_strict(hv_map, default=0)
              .cast(pl.Int32).alias("hvfhs_code"),

            pl.col("dispatching_base_num")
              .cast(pl.Utf8).fill_null("UNK")
              .replace_strict(disp_map, default=0)
              .cast(pl.Int32).alias("dispatch_code"),

            pl.col("originating_base_num")
              .cast(pl.Utf8).fill_null("UNK")
              .replace_strict(orig_map, default=0)
              .cast(pl.Int32).alias("origin_code"),

            # join keys
            pl.col("request_datetime").dt.truncate("1h").cast(pl.Datetime("us")).alias("ts_hour"),
            pl.col("request_datetime").dt.date().cast(pl.Date).alias("date"),
        ])
        # basic filtering
        .filter(
            (pl.col("y_time") > 0) &
            (pl.col("y_cost_no_tips") >= 0) &
            pl.col("request_datetime").is_not_null() &
            pl.col("PULocationID").is_not_null() &
            pl.col("DOLocationID").is_not_null() &
            pl.col("trip_miles").is_not_null() &
            (pl.col("trip_miles") >= 0)
        )
        # weather join
        .join(weather_lf, on="ts_hour", how="left")
        .with_columns([
            # fill numeric weather with 0; keep code with -1 for "missing"
            pl.col("temperature_2m").fill_null(0.0).cast(pl.Float32).alias("temperature_2m"),
            pl.col("precipitation").fill_null(0.0).cast(pl.Float32).alias("precipitation"),
            pl.col("wind_speed_10m").fill_null(0.0).cast(pl.Float32).alias("wind_speed_10m"),
            pl.col("weather_code").fill_null(-1).cast(pl.Int32).alias("weather_code"),

            (pl.col("precipitation") > 0.0).cast(pl.Int8).alias("is_precip"),
            (pl.col("weather_code") == -1).cast(pl.Int8).alias("weather_missing"),
        ])
        # holiday join
        .join(holiday_lf, on="date", how="left")
        .with_columns([
            # 关键：先 cast Int8 再 fill_null(0)，彻底避免 [bool, dyn int] 报错
            pl.col("is_holiday").cast(pl.Int8).fill_null(0).alias("is_holiday"),
            pl.col("is_before_holiday").cast(pl.Int8).fill_null(0).alias("is_before_holiday"),
            pl.col("is_after_holiday").cast(pl.Int8).fill_null(0).alias("is_after_holiday"),
        ])
    )

    # ---- final output schema (训练脚本按这个写死) ----
    out_cols = [
        "y_cost_no_tips", "y_time",

        "hour", "dow", "is_weekend", "month", "is_peak",
        "PULocationID", "DOLocationID", "same_zone", "route_id",

        "trip_miles", "log1p_trip_miles", "is_zero_miles",

        "shared_request_flag", "wav_request_flag", "access_a_ride_flag",
        "originating_base_missing",

        "hvfhs_code", "dispatch_code", "origin_code",

        "temperature_2m", "precipitation", "wind_speed_10m", "weather_code",
        "is_precip", "weather_missing",

        "is_holiday", "is_before_holiday", "is_after_holiday",
    ]

    collect_streaming(feats.select(out_cols)).write_parquet(out_path, compression="zstd")
    return out_path


# ---------- main ----------
if __name__ == "__main__":
    hv_map, disp_map, orig_map = build_global_maps()

    for m in range(1, 13):
        out = build_month_features_wxhol(m, hv_map, disp_map, orig_map)
        print(f"[OK] features month={m:02d} -> {out}")

    # quick schema check
    p = os.path.join(FEATURE_DIR_OUT, "features_2024-01.parquet")
    schema_keys = list(pl.read_parquet_schema(p).keys())
    print("[SCHEMA CHECK] has is_zero_miles?", "is_zero_miles" in schema_keys)
    print("[SCHEMA CHECK] n_cols=", len(schema_keys))


[maps] hvfhs=3 dispatch=3 originating=10
[OK] features month=01 -> /data/tlc_hvfhv_2024_features_wxhol/features_2024-01.parquet
[OK] features month=02 -> /data/tlc_hvfhv_2024_features_wxhol/features_2024-02.parquet
[OK] features month=03 -> /data/tlc_hvfhv_2024_features_wxhol/features_2024-03.parquet
[OK] features month=04 -> /data/tlc_hvfhv_2024_features_wxhol/features_2024-04.parquet
[OK] features month=05 -> /data/tlc_hvfhv_2024_features_wxhol/features_2024-05.parquet
[OK] features month=06 -> /data/tlc_hvfhv_2024_features_wxhol/features_2024-06.parquet
[OK] features month=07 -> /data/tlc_hvfhv_2024_features_wxhol/features_2024-07.parquet
[OK] features month=08 -> /data/tlc_hvfhv_2024_features_wxhol/features_2024-08.parquet
[OK] features month=09 -> /data/tlc_hvfhv_2024_features_wxhol/features_2024-09.parquet
[OK] features month=10 -> /data/tlc_hvfhv_2024_features_wxhol/features_2024-10.parquet
[OK] features month=11 -> /data/tlc_hvfhv_2024_features_wxhol/features_2024-11.parquet
[O

In [41]:
!pwd
!ln -s /data/tlc_hvfhv_2024_out ./tlc_hvfhv_2024_out
!ls -lah ./tlc_hvfhv_2024_out | head


/root
lrwxrwxrwx 1 root root 24 Jan 11 14:16 ./tlc_hvfhv_2024_out -> /data/tlc_hvfhv_2024_out


In [42]:
!free -g

               total        used        free      shared  buff/cache   available
Mem:            1007          34         114           0         858         981
Swap:              0           0           0


In [43]:
# import pandas as pd
# import holidays

# OUT_PATH = "/data/tlc_hvfhv_2024_out/holidays_NY_2024.parquet"

# hcal = holidays.US(subdiv="NY", years=[2024])  # NY 州假期 :contentReference[oaicite:2]{index=2}

# dates = pd.date_range("2024-01-01", "2024-12-31", freq="D")
# dfh = pd.DataFrame({"date": dates.date})
# dfh["is_holiday"] = [d in hcal for d in dfh["date"]]
# dfh["holiday_name"] = [hcal.get(d, "") for d in dfh["date"]]

# # 前后一天（更有用）
# is_h = dfh["is_holiday"].to_numpy()
# dfh["is_before_holiday"] = pd.Series(is_h).shift(-1, fill_value=False).to_numpy()
# dfh["is_after_holiday"]  = pd.Series(is_h).shift(+1, fill_value=False).to_numpy()

# dfh.to_parquet(OUT_PATH, index=False)
# print("saved:", OUT_PATH, "rows:", len(dfh))
# print(dfh[dfh["is_holiday"]].head())
# import pandas as pd
# import requests

# OUT_PATH = "/data/tlc_hvfhv_2024_out/weather_hourly_2024.parquet"

# def fetch_open_meteo_hourly_2024(
#     lat=40.7128, lon=-74.0060, timezone="America/New_York"
# ):
#     url = "https://archive-api.open-meteo.com/v1/archive"
#     params = {
#         "latitude": lat,
#         "longitude": lon,
#         "start_date": "2024-01-01",
#         "end_date": "2024-12-31",
#         "hourly": ",".join([
#             "temperature_2m",
#             "precipitation",
#             "wind_speed_10m",
#             "weather_code",
#         ]),
#         "timezone": timezone,
#     }

#     r = requests.get(url, params=params, timeout=120)
#     r.raise_for_status()
#     js = r.json()

#     df = pd.DataFrame({"ts_hour": pd.to_datetime(js["hourly"]["time"])})
#     for k, v in js["hourly"].items():
#         if k != "time":
#             df[k] = v

#     # 简单清理
#     df = df.sort_values("ts_hour").reset_index(drop=True)
#     return df

# dfw = fetch_open_meteo_hourly_2024()
# dfw.to_parquet(OUT_PATH, index=False)
# print("saved:", OUT_PATH, "rows:", len(dfw))
# print(dfw.head())


In [44]:
# import os
# import polars as pl

# DATA_DIR = "/data/tlc_hvfhv_2024"
# IN_TMPL  = "fhvhv_tripdata_2024-{m:02d}.parquet"

# FEATURE_DIR_OUT = "/data/tlc_hvfhv_2024_features_wxhol"
# os.makedirs(FEATURE_DIR_OUT, exist_ok=True)

# WEATHER_PATH = "/data/tlc_hvfhv_2024_out/weather_hourly_2024.parquet"
# HOLIDAY_PATH = "/data/tlc_hvfhv_2024_out/holidays_NY_2024.parquet"

# DATA_GLOB = f"{DATA_DIR}/fhvhv_tripdata_2024-*.parquet"

# def collect_streaming(lf: pl.LazyFrame) -> pl.DataFrame:
#     try:
#         return lf.collect(engine="streaming")
#     except TypeError:
#         return lf.collect()

# def yn01(colname: str) -> pl.Expr:
#     return (
#         pl.when(pl.col(colname) == "Y").then(1).otherwise(0)
#         .cast(pl.Int8)
#         .alias(colname)
#     )

# # ---------- 1) 全局 label encoding 映射（hvfhs/dispatch/origin） ----------
# def build_global_maps():
#     lf = pl.scan_parquet(DATA_GLOB).select([
#         pl.col("hvfhs_license_num").cast(pl.Utf8).fill_null("UNK").alias("hvfhs_license_num"),
#         pl.col("dispatching_base_num").cast(pl.Utf8).fill_null("UNK").alias("dispatching_base_num"),
#         pl.col("originating_base_num").cast(pl.Utf8).fill_null("UNK").alias("originating_base_num"),
#     ])
#     uniq = collect_streaming(lf.unique())

#     def make_map(col: str):
#         vals = uniq.get_column(col).unique().to_list()
#         vals = [v for v in vals if v != "UNK"]
#         vals.sort()
#         mp = {"UNK": 0}
#         for i, v in enumerate(vals, start=1):
#             mp[v] = i
#         return mp

#     hv_map = make_map("hvfhs_license_num")
#     disp_map = make_map("dispatching_base_num")
#     orig_map = make_map("originating_base_num")
#     print(f"[maps] hvfhs={len(hv_map)} dispatch={len(disp_map)} originating={len(orig_map)}")
#     return hv_map, disp_map, orig_map

# hv_map, disp_map, orig_map = build_global_maps()

# # ---------- 2) 维表 lazy scan ----------
# weather_lf = (
#     pl.scan_parquet(WEATHER_PATH)
#       .with_columns(pl.col("ts_hour").cast(pl.Datetime("us")))
#       .select(["ts_hour", "temperature_2m", "precipitation", "wind_speed_10m", "weather_code"])
# )


# holiday_lf = (
#     pl.scan_parquet(HOLIDAY_PATH)
#       .with_columns(pl.col("date").cast(pl.Date))
#       .select(["date", "is_holiday", "is_before_holiday", "is_after_holiday"])
# )

# # ---------- 3) 逐月生成 features（含天气/假期） ----------
# def month_overview(m: int) -> pl.DataFrame:
#     path = os.path.join(DATA_DIR, IN_TMPL.format(m=m))

#     use_cols = [
#         "request_datetime",
#         "PULocationID", "DOLocationID",
#         "trip_miles", "trip_time",
#         "hvfhs_license_num",
#         "dispatching_base_num",
#         "originating_base_num",
#         "shared_request_flag", "wav_request_flag", "access_a_ride_flag",
#         "base_passenger_fare", "tolls", "bcf", "sales_tax", "congestion_surcharge", "airport_fee",
#         "tips",
#     ]

#     lf = pl.scan_parquet(path).select(use_cols)

#     # 不含tips的开销标签（缺失当0）
#     y_cost = (
#         pl.col("base_passenger_fare").fill_null(0) +
#         pl.col("tolls").fill_null(0) +
#         pl.col("bcf").fill_null(0) +
#         pl.col("sales_tax").fill_null(0) +
#         pl.col("congestion_surcharge").fill_null(0) +
#         pl.col("airport_fee").fill_null(0)
#     ).alias("y_cost_no_tips")

#     lf = lf.with_columns([y_cost])

#     # 主聚合（全样本）
#     agg = lf.select([
#         pl.len().alias("n_rows"),

#         pl.col("request_datetime").min().alias("min_request_dt"),
#         pl.col("request_datetime").max().alias("max_request_dt"),

#         # null counts
#         pl.col("originating_base_num").null_count().alias("originating_base_nulls"),
#         pl.col("dispatching_base_num").null_count().alias("dispatching_base_nulls"),
#         pl.col("hvfhs_license_num").null_count().alias("hvfhs_nulls"),
#         pl.col("PULocationID").null_count().alias("PU_nulls"),
#         pl.col("DOLocationID").null_count().alias("DO_nulls"),
#         pl.col("trip_miles").null_count().alias("trip_miles_nulls"),
#         pl.col("trip_time").null_count().alias("trip_time_nulls"),
#         pl.col("y_cost_no_tips").null_count().alias("cost_nulls"),
#         pl.col("tips").null_count().alias("tips_nulls"),

#         # numeric summary
#         pl.col("trip_miles").mean().alias("trip_miles_mean"),
#         pl.col("trip_miles").std().alias("trip_miles_std"),
#         pl.col("trip_miles").min().alias("trip_miles_min"),
#         pl.col("trip_miles").max().alias("trip_miles_max"),

#         # ✅ trip_miles == 0 计数（保留在主表里）
#         (pl.col("trip_miles") == 0).sum().alias("trip_miles_zero_cnt"),

#         pl.col("trip_time").mean().alias("trip_time_mean"),
#         pl.col("trip_time").std().alias("trip_time_std"),
#         pl.col("trip_time").min().alias("trip_time_min"),
#         pl.col("trip_time").max().alias("trip_time_max"),

#         pl.col("y_cost_no_tips").mean().alias("cost_mean"),
#         pl.col("y_cost_no_tips").std().alias("cost_std"),
#         pl.col("y_cost_no_tips").min().alias("cost_min"),
#         pl.col("y_cost_no_tips").max().alias("cost_max"),

#         pl.col("tips").mean().alias("tips_mean"),
#         pl.col("tips").min().alias("tips_min"),
#         pl.col("tips").max().alias("tips_max"),
#     ])

#     # ✅ 0里程子样本画像（注意：不要再输出 trip_miles_zero_cnt 同名列）
#     zero_agg = lf.filter(pl.col("trip_miles") == 0).select([
#         pl.col("trip_time").mean().alias("trip_time_mean_when_miles0"),
#         pl.col("y_cost_no_tips").mean().alias("cost_mean_when_miles0"),
#         pl.col("tips").mean().alias("tips_mean_when_miles0"),
#     ])

#     df_main = collect_streaming(agg)
#     df_zero = collect_streaming(zero_agg)

#     # 横向合并（不会重名）
#     df = pl.concat([df_main, df_zero], how="horizontal").with_columns([
#         pl.lit(f"2024-{m:02d}").alias("month"),
#     ])

#     # ✅ 0里程占比
#     df = df.with_columns([
#         (pl.col("trip_miles_zero_cnt") / pl.col("n_rows")).alias("trip_miles_zero_rate"),
#     ])

#     # ✅ 统一时间精度，避免 concat 报错（μs vs ns）
#     df = df.with_columns([
#         pl.col("min_request_dt").cast(pl.Datetime("ns")),
#         pl.col("max_request_dt").cast(pl.Datetime("ns")),
#     ])

#     # ✅ 缺失率（null_rate = nulls / n_rows）
#     df = df.with_columns([
#         (pl.col("originating_base_nulls") / pl.col("n_rows")).alias("originating_base_null_rate"),
#         (pl.col("dispatching_base_nulls") / pl.col("n_rows")).alias("dispatching_base_null_rate"),
#         (pl.col("hvfhs_nulls") / pl.col("n_rows")).alias("hvfhs_null_rate"),
#         (pl.col("PU_nulls") / pl.col("n_rows")).alias("PU_null_rate"),
#         (pl.col("DO_nulls") / pl.col("n_rows")).alias("DO_null_rate"),
#         (pl.col("trip_miles_nulls") / pl.col("n_rows")).alias("trip_miles_null_rate"),
#         (pl.col("trip_time_nulls") / pl.col("n_rows")).alias("trip_time_null_rate"),
#         (pl.col("cost_nulls") / pl.col("n_rows")).alias("cost_null_rate"),
#         (pl.col("tips_nulls") / pl.col("n_rows")).alias("tips_null_rate"),
#     ])

#     # cast float32（省空间）
#     float_cols = [
#         "trip_miles_mean","trip_miles_std","trip_miles_min","trip_miles_max",
#         "trip_time_mean","trip_time_std","trip_time_min","trip_time_max",
#         "cost_mean","cost_std","cost_min","cost_max",
#         "tips_mean","tips_min","tips_max",
#         "originating_base_null_rate","dispatching_base_null_rate","hvfhs_null_rate",
#         "PU_null_rate","DO_null_rate","trip_miles_null_rate","trip_time_null_rate",
#         "cost_null_rate","tips_null_rate",
#         "trip_time_mean_when_miles0","cost_mean_when_miles0","tips_mean_when_miles0",
#         "trip_miles_zero_rate",
#     ]
#     df = df.with_columns([pl.col(c).cast(pl.Float32) for c in float_cols])

#     return df


# for m in range(1, 13):
#     build_month_features_wxhol(m)


[maps] hvfhs=3 dispatch=3 originating=10


/tmp/ipykernel_939/89810785.py:136: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col("hvfhs_license_num").cast(pl.Utf8).fill_null("UNK").replace(hv_map, default=0).cast(pl.Int16).alias("hvfhs_code"),
/tmp/ipykernel_939/89810785.py:137: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col("dispatching_base_num").cast(pl.Utf8).fill_null("UNK").replace(disp_map, default=0).cast(pl.Int16).alias("dispatch_code"),
/tmp/ipykernel_939/89810785.py:138: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col("originating_base_num").cast(pl.Utf8).fill_null("UNK").replace(orig_map, default=0).cast(pl.Int16).alias("or

[OK] wxhol features month=01 -> /data/tlc_hvfhv_2024_features_wxhol/features_2024-01.parquet
[OK] wxhol features month=02 -> /data/tlc_hvfhv_2024_features_wxhol/features_2024-02.parquet
[OK] wxhol features month=03 -> /data/tlc_hvfhv_2024_features_wxhol/features_2024-03.parquet
[OK] wxhol features month=04 -> /data/tlc_hvfhv_2024_features_wxhol/features_2024-04.parquet
[OK] wxhol features month=05 -> /data/tlc_hvfhv_2024_features_wxhol/features_2024-05.parquet
[OK] wxhol features month=06 -> /data/tlc_hvfhv_2024_features_wxhol/features_2024-06.parquet
[OK] wxhol features month=07 -> /data/tlc_hvfhv_2024_features_wxhol/features_2024-07.parquet
[OK] wxhol features month=08 -> /data/tlc_hvfhv_2024_features_wxhol/features_2024-08.parquet
[OK] wxhol features month=09 -> /data/tlc_hvfhv_2024_features_wxhol/features_2024-09.parquet
[OK] wxhol features month=10 -> /data/tlc_hvfhv_2024_features_wxhol/features_2024-10.parquet
[OK] wxhol features month=11 -> /data/tlc_hvfhv_2024_features_wxhol/fe

In [3]:
import polars as pl
df = pl.read_parquet("/data/tlc_hvfhv_2024_features_wxhol/features_2024-01.parquet", columns=["weather_missing","is_holiday"])
print(df.select([
    pl.len().alias("n"),
    pl.col("weather_missing").mean().alias("weather_missing_rate"),
    pl.col("is_holiday").mean().alias("holiday_rate"),
]))


shape: (1, 3)
┌──────────┬──────────────────────┬──────────────┐
│ n        ┆ weather_missing_rate ┆ holiday_rate │
│ ---      ┆ ---                  ┆ ---          │
│ u32      ┆ f64                  ┆ f64          │
╞══════════╪══════════════════════╪══════════════╡
│ 19663863 ┆ 0.0                  ┆ 0.0609       │
└──────────┴──────────────────────┴──────────────┘


In [4]:
import os
import gc
import numpy as np
import polars as pl
import xgboost as xgb
import lightgbm as lgb

FEATURE_DIR = "/data/tlc_hvfhv_2024_features_wxhol"
MODEL_DIR   = "/data/tlc_hvfhv_2024_models"
os.makedirs(MODEL_DIR, exist_ok=True)

TRAIN_MONTHS = list(range(1, 10))   # 1-9
VALID_MONTHS = [10]
TEST_MONTHS  = [11, 12]

Y_COST_COL = "y_cost_no_tips"
Y_TIME_COL = "y_time"

# ===== 基础特征列 =====
BASE_FEAT_COLS = [
    "hour", "dow", "is_weekend", "month", "is_peak",
    "PULocationID", "DOLocationID", "same_zone", "route_id",
    "trip_miles", "log1p_trip_miles",
    "shared_request_flag", "wav_request_flag", "access_a_ride_flag",
    "originating_base_missing",
    "hvfhs_code", "dispatch_code", "origin_code",
]

# ===== 天气 + 假期 =====
EXTRA_FEAT_COLS = [
    "temperature_2m", "precipitation", "wind_speed_10m", "weather_code",
    "is_precip", "weather_missing",
    "is_holiday", "is_before_holiday", "is_after_holiday",
    "is_zero_miles",
]

FEAT_COLS = BASE_FEAT_COLS + EXTRA_FEAT_COLS

# 这些列先按整型/布尔型 cast，再统一转 float32 供模型用
INT_COLS = ["PULocationID", "DOLocationID", "route_id", "hvfhs_code", "dispatch_code", "origin_code", "weather_code",
            "hour", "dow", "month"]
BIN_COLS = ["same_zone","is_weekend","is_peak",
            "shared_request_flag","wav_request_flag","access_a_ride_flag","originating_base_missing",
            "is_precip","weather_missing","is_holiday","is_before_holiday","is_after_holiday",
            "is_zero_miles",]
FLOAT_COLS = ["trip_miles","log1p_trip_miles","temperature_2m","precipitation","wind_speed_10m"]

def month_path(m: int) -> str:
    return os.path.join(FEATURE_DIR, f"features_2024-{m:02d}.parquet")

def collect_streaming(lf: pl.LazyFrame) -> pl.DataFrame:
    try:
        return lf.collect(engine="streaming")
    except TypeError:
        return lf.collect()

# --- 用训练月估计 clip bounds：抽样分位数 ---
def estimate_clip_bounds(months, y_col: str, sample_pct: int = 2, q_low: float = 0.005, q_high: float = 0.995):
    paths = [month_path(m) for m in months]
    lf = pl.scan_parquet(paths).select(["route_id", y_col]).drop_nulls()
    lf_s = lf.filter(pl.col("route_id").hash(seed=42).mod(100) < sample_pct)
    df_s = collect_streaming(lf_s)

    y = df_s.get_column(y_col).to_numpy()
    y = y[np.isfinite(y)]
    if len(y) == 0:
        raise RuntimeError(f"Sample for {y_col} is empty. Check columns/data.")
    lo = float(np.quantile(y, q_low))
    hi = float(np.quantile(y, q_high))
    return lo, hi, len(y)

# --- 按月抽样读入：返回 float32 X, float32 y_log, float32 y_raw ---
def load_months_sample_numpy(months, cost_clip, time_clip, y_mode: str, sample_pct: int = 2):
    assert y_mode in ("cost", "time")
    X_list, ylog_list, yraw_list = [], [], []

    for m in months:
        path = month_path(m)

        lf = pl.scan_parquet(path).select(FEAT_COLS + [Y_COST_COL, Y_TIME_COL])

        lf = (
            lf.filter((pl.col(Y_COST_COL) >= 0) & (pl.col(Y_TIME_COL) > 0))
              .filter(pl.col("route_id").hash(seed=42).mod(100) < sample_pct)
              .with_columns([
                  pl.col(Y_COST_COL).clip(cost_clip[0], cost_clip[1]).log1p().alias("y_cost_log"),
                  pl.col(Y_TIME_COL).clip(time_clip[0], time_clip[1]).log1p().alias("y_time_log"),
              ])
              .select(FEAT_COLS + ["y_cost_log","y_time_log", Y_COST_COL, Y_TIME_COL])
        )

        df = collect_streaming(lf)

        # 先把离散/布尔/连续列 cast 到合理 dtype
        for c in INT_COLS:
            if c in df.columns:
                df = df.with_columns(pl.col(c).cast(pl.Int32))
        for c in BIN_COLS:
            if c in df.columns:
                df = df.with_columns(pl.col(c).cast(pl.Int8))
        for c in FLOAT_COLS:
            if c in df.columns:
                df = df.with_columns(pl.col(c).cast(pl.Float32))

        # ✅ 关键：所有特征统一转 float32，保证 to_numpy 得到 float32 ndarray（避免 object）
        df = df.with_columns([pl.col(c).cast(pl.Float32) for c in FEAT_COLS])

        X = df.select(FEAT_COLS).to_numpy()
        if y_mode == "cost":
            y_log = df["y_cost_log"].to_numpy().astype(np.float32, copy=False)
            y_raw = df[Y_COST_COL].to_numpy().astype(np.float32, copy=False)
        else:
            y_log = df["y_time_log"].to_numpy().astype(np.float32, copy=False)
            y_raw = df[Y_TIME_COL].to_numpy().astype(np.float32, copy=False)

        X = np.nan_to_num(X, nan=0.0).astype(np.float32, copy=False)

        X_list.append(X); ylog_list.append(y_log); yraw_list.append(y_raw)

        print(f"[OK] sample month={m:02d}, rows={X.shape[0]:,}")
        del df
        gc.collect()

    X_all = np.concatenate(X_list, axis=0)
    ylog_all = np.concatenate(ylog_list, axis=0)
    yraw_all = np.concatenate(yraw_list, axis=0)

    del X_list, ylog_list, yraw_list
    gc.collect()
    return X_all, ylog_all, yraw_all

# =========================
# XGBoost
# =========================
def train_xgb_regressor(X_tr, y_tr, X_va, y_va, model_name: str):
    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dvalid = xgb.DMatrix(X_va, label=y_va)

    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "learning_rate": 0.08,
        "max_depth": 8,
        "min_child_weight": 10,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "max_bin": 256,
        "lambda": 1.0,
        "alpha": 0.0,
        "seed": 42,
    }

    booster = xgb.train(
        params,
        dtrain,
        num_boost_round=2000,
        evals=[(dtrain, "train"), (dvalid, "valid")],
        early_stopping_rounds=120,
        verbose_eval=50,
    )

    out_path = os.path.join(MODEL_DIR, f"{model_name}.json")
    booster.save_model(out_path)
    print("[SAVED]", out_path)
    return booster

def eval_xgb_rmse_mae(booster, X, y_true_raw):
    d = xgb.DMatrix(X)
    best = booster.best_iteration
    pred_log = booster.predict(d, iteration_range=(0, best + 1))
    pred = np.expm1(pred_log)
    y_true = y_true_raw.astype(float)
    rmse = float(np.sqrt(np.mean((pred - y_true) ** 2)))
    mae  = float(np.mean(np.abs(pred - y_true)))
    return rmse, mae

# =========================
# LightGBM
# =========================
def train_lgb_regressor(X_tr, y_tr, X_va, y_va, model_name: str):
    lgb_train = lgb.Dataset(X_tr, label=y_tr, free_raw_data=False)
    lgb_valid = lgb.Dataset(X_va, label=y_va, reference=lgb_train, free_raw_data=False)

    params = {
        "objective": "regression",
        "metric": "rmse",
        "learning_rate": 0.06,
        "num_leaves": 128,
        "min_data_in_leaf": 80,
        "feature_fraction": 0.9,
        "bagging_fraction": 0.9,
        "bagging_freq": 1,
        "max_bin": 255,
        "verbosity": -1,
        "seed": 42,
    }

    booster = lgb.train(
        params,
        lgb_train,
        num_boost_round=5000,
        valid_sets=[lgb_train, lgb_valid],
        valid_names=["train", "valid"],
        callbacks=[
            lgb.early_stopping(stopping_rounds=200),
            lgb.log_evaluation(period=50),
        ],
    )

    out_path = os.path.join(MODEL_DIR, f"{model_name}.txt")
    booster.save_model(out_path)
    print("[SAVED]", out_path)
    return booster

def eval_lgb_rmse_mae(booster, X, y_true_raw):
    pred_log = booster.predict(X, num_iteration=booster.best_iteration)
    pred = np.expm1(pred_log)
    y_true = y_true_raw.astype(float)
    rmse = float(np.sqrt(np.mean((pred - y_true) ** 2)))
    mae  = float(np.mean(np.abs(pred - y_true)))
    return rmse, mae

# =======================
# 主流程
# =======================
cost_lo, cost_hi, n1 = estimate_clip_bounds(TRAIN_MONTHS, Y_COST_COL, sample_pct=2, q_low=0.005, q_high=0.995)
time_lo, time_hi, n2 = estimate_clip_bounds(TRAIN_MONTHS, Y_TIME_COL, sample_pct=2, q_low=0.005, q_high=0.995)

print(f"[clip] cost_no_tips: lo={cost_lo:.4f}, hi={cost_hi:.4f}, sample_n={n1:,}")
print(f"[clip] trip_time:    lo={time_lo:.4f}, hi={time_hi:.4f}, sample_n={n2:,}")

cost_clip = (cost_lo, cost_hi)
time_clip = (time_lo, time_hi)

SAMPLE_TRAIN = 5
SAMPLE_VALID = 2
SAMPLE_TEST  = 2

def run_one_target(y_mode: str):
    # --- load ---
    X_tr, ytr_log, ytr_raw = load_months_sample_numpy(TRAIN_MONTHS, cost_clip, time_clip, y_mode=y_mode, sample_pct=SAMPLE_TRAIN)
    X_va, yva_log, yva_raw = load_months_sample_numpy(VALID_MONTHS, cost_clip, time_clip, y_mode=y_mode, sample_pct=SAMPLE_VALID)
    X_te, yte_log, yte_raw = load_months_sample_numpy(TEST_MONTHS,  cost_clip, time_clip, y_mode=y_mode, sample_pct=SAMPLE_TEST)

    print(f"[rows] {y_mode} train={len(ytr_log):,}, valid={len(yva_log):,}, test={len(yte_log):,}")

    # --- XGBoost ---
    xgb_name = f"xgb_{y_mode}_log_wxhol"
    model_xgb = train_xgb_regressor(X_tr, ytr_log, X_va, yva_log, model_name=xgb_name)
    rmse_x_va, mae_x_va = eval_xgb_rmse_mae(model_xgb, X_va, yva_raw)
    rmse_x_te, mae_x_te = eval_xgb_rmse_mae(model_xgb, X_te, yte_raw)

    # --- LightGBM ---
    lgb_name = f"lgb_{y_mode}_log_wxhol"
    model_lgb = train_lgb_regressor(X_tr, ytr_log, X_va, yva_log, model_name=lgb_name)
    rmse_l_va, mae_l_va = eval_lgb_rmse_mae(model_lgb, X_va, yva_raw)
    rmse_l_te, mae_l_te = eval_lgb_rmse_mae(model_lgb, X_te, yte_raw)

    print(f"\n=== {y_mode.upper()} (raw-scale eval) ===")
    print(f"[XGB] Valid RMSE={rmse_x_va:.4f}, MAE={mae_x_va:.4f} | Test RMSE={rmse_x_te:.4f}, MAE={mae_x_te:.4f}")
    print(f"[LGB] Valid RMSE={rmse_l_va:.4f}, MAE={mae_l_va:.4f} | Test RMSE={rmse_l_te:.4f}, MAE={mae_l_te:.4f}")

    # 清内存
    del X_tr, ytr_log, ytr_raw, X_va, yva_log, yva_raw, X_te, yte_log, yte_raw
    gc.collect()

# 两个目标都跑
run_one_target("cost")
run_one_target("time")


[clip] cost_no_tips: lo=7.2200, hi=119.7900, sample_n=3,450,955
[clip] trip_time:    lo=173.0000, hi=4276.0000, sample_n=3,450,955
[OK] sample month=01, rows=941,074
[OK] sample month=02, rows=922,748
[OK] sample month=03, rows=1,010,996
[OK] sample month=04, rows=939,736
[OK] sample month=05, rows=981,461
[OK] sample month=06, rows=951,837
[OK] sample month=07, rows=910,995
[OK] sample month=08, rows=911,799
[OK] sample month=09, rows=910,595
[OK] sample month=10, rows=382,503
[OK] sample month=11, rows=387,473
[OK] sample month=12, rows=407,435
[rows] cost train=8,481,241, valid=382,503, test=794,908
[0]	train-rmse:0.56060	valid-rmse:0.54954
[50]	train-rmse:0.22758	valid-rmse:0.24321
[100]	train-rmse:0.21011	valid-rmse:0.22465
[150]	train-rmse:0.20422	valid-rmse:0.21777
[200]	train-rmse:0.20107	valid-rmse:0.21484
[250]	train-rmse:0.19900	valid-rmse:0.21271
[300]	train-rmse:0.19764	valid-rmse:0.21127
[350]	train-rmse:0.19651	valid-rmse:0.21024
[400]	train-rmse:0.19552	valid-rmse:0.209

将训练抽样比例从 2% 提升至 5% 后，验证/测试指标更稳定，尤其在测试集上 RMSE 与 MAE 有明显改善，说明模型在更多样本上学习到了更稳健的时空与天气交互模式，并降低了抽样噪声带来的波动。

In [5]:

# =========================
# LightGBM提升版
# =========================
def train_lgb_regressor(X_tr, y_tr, X_va, y_va, model_name: str):
    lgb_train = lgb.Dataset(X_tr, label=y_tr, free_raw_data=False)
    lgb_valid = lgb.Dataset(X_va, label=y_va, reference=lgb_train, free_raw_data=False)

    params = {
        "objective": "regression",
        "metric": "rmse",
        "learning_rate": 0.10,
        "num_leaves": 96,
        "min_data_in_leaf": 150,
        "feature_fraction": 0.9,
        "bagging_fraction": 0.9,
        "bagging_freq": 1,
        "max_bin": 255,
        "verbosity": -1,
        "seed": 42,
    }
    # num_boost_round=2500


    booster = lgb.train(
        params,
        lgb_train,
        num_boost_round=5000,
        valid_sets=[lgb_train, lgb_valid],
        valid_names=["train", "valid"],
        callbacks=[
            lgb.early_stopping(stopping_rounds=200),
            lgb.log_evaluation(period=50),
        ],
    )

    out_path = os.path.join(MODEL_DIR, f"{model_name}.txt")
    booster.save_model(out_path)
    print("[SAVED]", out_path)
    return booster

def eval_lgb_rmse_mae(booster, X, y_true_raw):
    pred_log = booster.predict(X, num_iteration=booster.best_iteration)
    pred = np.expm1(pred_log)
    y_true = y_true_raw.astype(float)
    rmse = float(np.sqrt(np.mean((pred - y_true) ** 2)))
    mae  = float(np.mean(np.abs(pred - y_true)))
    return rmse, mae

# =======================
# 主流程
# =======================
cost_lo, cost_hi, n1 = estimate_clip_bounds(TRAIN_MONTHS, Y_COST_COL, sample_pct=2, q_low=0.005, q_high=0.995)
time_lo, time_hi, n2 = estimate_clip_bounds(TRAIN_MONTHS, Y_TIME_COL, sample_pct=2, q_low=0.005, q_high=0.995)

print(f"[clip] cost_no_tips: lo={cost_lo:.4f}, hi={cost_hi:.4f}, sample_n={n1:,}")
print(f"[clip] trip_time:    lo={time_lo:.4f}, hi={time_hi:.4f}, sample_n={n2:,}")

cost_clip = (cost_lo, cost_hi)
time_clip = (time_lo, time_hi)

SAMPLE_TRAIN = 5
SAMPLE_VALID = 2
SAMPLE_TEST  = 2

def run_one_target(y_mode: str):
    # --- load ---
    X_tr, ytr_log, ytr_raw = load_months_sample_numpy(TRAIN_MONTHS, cost_clip, time_clip, y_mode=y_mode, sample_pct=SAMPLE_TRAIN)
    X_va, yva_log, yva_raw = load_months_sample_numpy(VALID_MONTHS, cost_clip, time_clip, y_mode=y_mode, sample_pct=SAMPLE_VALID)
    X_te, yte_log, yte_raw = load_months_sample_numpy(TEST_MONTHS,  cost_clip, time_clip, y_mode=y_mode, sample_pct=SAMPLE_TEST)

    print(f"[rows] {y_mode} train={len(ytr_log):,}, valid={len(yva_log):,}, test={len(yte_log):,}")

    # --- XGBoost ---
    xgb_name = f"xgb_{y_mode}_log_wxhol"
    model_xgb = train_xgb_regressor(X_tr, ytr_log, X_va, yva_log, model_name=xgb_name)
    rmse_x_va, mae_x_va = eval_xgb_rmse_mae(model_xgb, X_va, yva_raw)
    rmse_x_te, mae_x_te = eval_xgb_rmse_mae(model_xgb, X_te, yte_raw)

    # --- LightGBM ---
    lgb_name = f"lgb_{y_mode}_log_wxhol"
    model_lgb = train_lgb_regressor(X_tr, ytr_log, X_va, yva_log, model_name=lgb_name)
    rmse_l_va, mae_l_va = eval_lgb_rmse_mae(model_lgb, X_va, yva_raw)
    rmse_l_te, mae_l_te = eval_lgb_rmse_mae(model_lgb, X_te, yte_raw)

    print(f"\n=== {y_mode.upper()} (raw-scale eval) ===")
    print(f"[XGB] Valid RMSE={rmse_x_va:.4f}, MAE={mae_x_va:.4f} | Test RMSE={rmse_x_te:.4f}, MAE={mae_x_te:.4f}")
    print(f"[LGB] Valid RMSE={rmse_l_va:.4f}, MAE={mae_l_va:.4f} | Test RMSE={rmse_l_te:.4f}, MAE={mae_l_te:.4f}")

    # 清内存
    del X_tr, ytr_log, ytr_raw, X_va, yva_log, yva_raw, X_te, yte_log, yte_raw
    gc.collect()

# 两个目标都跑
run_one_target("cost")
run_one_target("time")


[clip] cost_no_tips: lo=7.2200, hi=119.7900, sample_n=3,450,955
[clip] trip_time:    lo=173.0000, hi=4276.0000, sample_n=3,450,955
[OK] sample month=01, rows=941,074
[OK] sample month=02, rows=922,748
[OK] sample month=03, rows=1,010,996
[OK] sample month=04, rows=939,736
[OK] sample month=05, rows=981,461
[OK] sample month=06, rows=951,837
[OK] sample month=07, rows=910,995
[OK] sample month=08, rows=911,799
[OK] sample month=09, rows=910,595
[OK] sample month=10, rows=382,503
[OK] sample month=11, rows=387,473
[OK] sample month=12, rows=407,435
[rows] cost train=8,481,241, valid=382,503, test=794,908
[0]	train-rmse:0.56060	valid-rmse:0.54954
[50]	train-rmse:0.22758	valid-rmse:0.24321
[100]	train-rmse:0.21011	valid-rmse:0.22465
[150]	train-rmse:0.20422	valid-rmse:0.21777
[200]	train-rmse:0.20107	valid-rmse:0.21484
[250]	train-rmse:0.19900	valid-rmse:0.21271
[300]	train-rmse:0.19764	valid-rmse:0.21127
[350]	train-rmse:0.19651	valid-rmse:0.21024
[400]	train-rmse:0.19552	valid-rmse:0.209

训练阶段我们采用“**按时间切分 + 抽样训练 + 目标稳健变换**”的处理流程，以兼顾计算可行性与结果可解释性。首先，数据按月份进行时间序列切分：使用 2024 年 1–9 月作为训练集、10 月作为验证集、11–12 月作为测试集，确保评估过程符合真实业务中的时间顺序，避免未来信息泄露。由于全年 HVFHV 数据规模达到数亿级别，直接全量加载到内存并训练会导致资源消耗过大且迭代效率低，因此我们在每个月数据上采用基于 `route_id` 的哈希过滤实现近似随机抽样（例如抽取 1%–5%），在保持样本覆盖面与分布代表性的同时显著降低训练开销，使得多次调参与特征对比实验在合理时间内完成。

在建模目标方面，`y_cost_no_tips`（不含小费的乘客开销）与 `y_time`（行程时长）均呈现明显的右偏长尾分布，极端值会使平方误差损失被少量异常样本主导，导致模型不稳定。为提高鲁棒性，我们仅在训练集抽样数据上估计分位数阈值（如 0.5% 与 99.5%），对目标进行 winsorization 截断，抑制异常极端值的影响；随后对截断后的目标采用 `log1p` 变换（即 (\log(1+y))），把长尾分布压缩到更接近对称的尺度，使模型更关注相对误差并提升收敛稳定性。训练时使用变换后的目标进行回归，推断时再用 `expm1` 将预测值还原到原始尺度，并在验证集与测试集上以原尺度计算 RMSE/MAE 进行性能评估。模型选择上，我们采用基于梯度提升树的 XGBoost（`hist` 树算法）进行训练，能够高效处理大规模样本并捕捉非线性关系，同时保持较好的可解释性（例如特征重要性分析），便于说明各类特征（时间、空间、里程、平台属性、天气与假期因素）对费用与时长预测的贡献。


在基线 LightGBM 参数下，模型在验证集上长期保持微弱改进，容易出现训练轮数过长的问题。为提升训练效率并降低过拟合风险，我们采用“更快更稳”的参数配置：提高学习率（加快收敛）、降低叶子数并提高叶节点最小样本数（限制模型复杂度）、同时配合 early stopping 在验证集无显著提升时提前终止训练。实证结果表明，该策略在费用预测任务中可显著减少训练轮数且性能保持稳定；在时长预测任务中，验证集仍存在缓慢改进，说明目标噪声更强，后续可通过进一步增加正则或收紧轮数上限获得更好的训练效率。

In [6]:
# import os, gc
# import numpy as np
# import polars as pl
# from sklearn.impute import SimpleImputer

# from sklearn.pipeline import Pipeline
# from sklearn.preprocessing import StandardScaler
# from sklearn.linear_model import Ridge
# from sklearn.ensemble import RandomForestRegressor

# FEATURE_DIR = "/data/tlc_hvfhv_2024_features_wxhol"
# MODEL_DIR   = "/data/tlc_hvfhv_2024_models"
# os.makedirs(MODEL_DIR, exist_ok=True)

# TRAIN_MONTHS = list(range(1, 10))   # 1-9
# VALID_MONTHS = [10]
# TEST_MONTHS  = [11, 12]

# Y_COST_COL = "y_cost_no_tips"
# Y_TIME_COL = "y_time"

# # ===== 全量特征（给 RF 用，树模型吃得下整数编码特征）=====
# FEAT_COLS_FULL = [
#     "hour", "dow", "is_weekend", "month", "is_peak",
#     "PULocationID", "DOLocationID", "same_zone", "route_id",
#     "trip_miles", "log1p_trip_miles",
#     "shared_request_flag", "wav_request_flag", "access_a_ride_flag",
#     "originating_base_missing",
#     "hvfhs_code", "dispatch_code", "origin_code",
#     # weather + holiday
#     "temperature_2m", "precipitation", "wind_speed_10m", "weather_code",
#     "is_precip", "weather_missing",
#     "is_holiday", "is_before_holiday", "is_after_holiday",
# ]

# # ===== Ridge 用“低维/连续/少量离散”特征（避免高基数造成线性模型无意义/太难处理）=====
# # 这里我刻意不放 route_id / PULocationID / DOLocationID（它们对线性模型需要 one-hot/hash 才像样，太重）
# FEAT_COLS_RIDGE = [
#     "hour", "dow", "is_weekend", "month", "is_peak",
#     "trip_miles", "log1p_trip_miles",
#     "shared_request_flag", "wav_request_flag", "access_a_ride_flag",
#     "originating_base_missing",
#     "hvfhs_code", "dispatch_code", "origin_code",
#     "temperature_2m", "precipitation", "wind_speed_10m", "weather_code",
#     "is_precip", "weather_missing",
#     "is_holiday", "is_before_holiday", "is_after_holiday",
# ]

# INT_COLS = ["hvfhs_code", "dispatch_code", "origin_code", "weather_code", "hour", "dow", "month"]
# BIN_COLS = [
#     "is_weekend","is_peak","shared_request_flag","wav_request_flag","access_a_ride_flag",
#     "originating_base_missing","is_precip","weather_missing","is_holiday","is_before_holiday","is_after_holiday",
#     "same_zone"
# ]
# FLOAT_COLS = ["trip_miles","log1p_trip_miles","temperature_2m","precipitation","wind_speed_10m"]

# def month_path(m: int) -> str:
#     return os.path.join(FEATURE_DIR, f"features_2024-{m:02d}.parquet")

# def collect_streaming(lf: pl.LazyFrame) -> pl.DataFrame:
#     try:
#         return lf.collect(engine="streaming")
#     except TypeError:
#         return lf.collect()

# def estimate_clip_bounds(months, y_col: str, sample_pct: int = 2, q_low: float = 0.005, q_high: float = 0.995):
#     paths = [month_path(m) for m in months]
#     lf = pl.scan_parquet(paths).select(["route_id", y_col]).drop_nulls()
#     lf_s = lf.filter(pl.col("route_id").hash(seed=42).mod(100) < sample_pct)
#     df_s = collect_streaming(lf_s)

#     y = df_s.get_column(y_col).to_numpy()
#     y = y[np.isfinite(y)]
#     lo = float(np.quantile(y, q_low))
#     hi = float(np.quantile(y, q_high))
#     return lo, hi, len(y)

# def load_sample_numpy(months, feat_cols, cost_clip, time_clip, y_mode: str, sample_pct: int):
#     """
#     y_mode: 'cost' or 'time'
#     返回: X, y_log, y_raw
#     """
#     assert y_mode in ("cost", "time")

#     X_list, ylog_list, yraw_list = [], [], []

#     for m in months:
#         path = month_path(m)

#         # 只读必要列
#         lf = pl.scan_parquet(path).select(["route_id"] + feat_cols + [Y_COST_COL, Y_TIME_COL])

#         # 过滤 + 抽样（route_id hash）
#         lf = (
#             lf.filter((pl.col(Y_COST_COL) >= 0) & (pl.col(Y_TIME_COL) > 0))
#               .filter(pl.col("route_id").hash(seed=42).mod(100) < sample_pct)
#               .with_columns([
#                   pl.col(Y_COST_COL).clip(cost_clip[0], cost_clip[1]).log1p().alias("y_cost_log"),
#                   pl.col(Y_TIME_COL).clip(time_clip[0], time_clip[1]).log1p().alias("y_time_log"),
#               ])
#               .select(feat_cols + ["y_cost_log","y_time_log", Y_COST_COL, Y_TIME_COL])
#         )

#         df = collect_streaming(lf)

#         # dtype 统一（省内存+稳）
#         for c in INT_COLS:
#             if c in df.columns:
#                 df = df.with_columns(pl.col(c).cast(pl.Int32))
#         for c in BIN_COLS:
#             if c in df.columns:
#                 df = df.with_columns(pl.col(c).cast(pl.Int8))
#         for c in FLOAT_COLS:
#             if c in df.columns:
#                 df = df.with_columns(pl.col(c).cast(pl.Float32))

#         X = df.select(feat_cols).to_numpy()

#         if y_mode == "cost":
#             y_log = df["y_cost_log"].to_numpy()
#             y_raw = df[Y_COST_COL].to_numpy()
#         else:
#             y_log = df["y_time_log"].to_numpy()
#             y_raw = df[Y_TIME_COL].to_numpy()

#         X_list.append(X); ylog_list.append(y_log); yraw_list.append(y_raw)
#         print(f"[OK] sample month={m:02d}, rows={X.shape[0]:,}")
#         del df
#         gc.collect()

#     X_all = np.concatenate(X_list, axis=0)
#     ylog_all = np.concatenate(ylog_list, axis=0)
#     yraw_all = np.concatenate(yraw_list, axis=0)
#     del X_list, ylog_list, yraw_list
#     gc.collect()
#     return X_all, ylog_all, yraw_all

# def eval_rmse_mae(pred_raw, y_true_raw):
#     pred_raw = pred_raw.astype(float)
#     y_true_raw = y_true_raw.astype(float)
#     rmse = float(np.sqrt(np.mean((pred_raw - y_true_raw) ** 2)))
#     mae  = float(np.mean(np.abs(pred_raw - y_true_raw)))
#     return rmse, mae

# # =========================
# # 1) clip 阈值（只用训练月抽样估计）
# # =========================
# cost_lo, cost_hi, n1 = estimate_clip_bounds(TRAIN_MONTHS, Y_COST_COL, sample_pct=2)
# time_lo, time_hi, n2 = estimate_clip_bounds(TRAIN_MONTHS, Y_TIME_COL, sample_pct=2)
# cost_clip = (cost_lo, cost_hi)
# time_clip = (time_lo, time_hi)

# print(f"[clip] cost: lo={cost_lo:.4f}, hi={cost_hi:.4f}, sample_n={n1:,}")
# print(f"[clip] time: lo={time_lo:.4f}, hi={time_hi:.4f}, sample_n={n2:,}")

# # =========================
# # 2) Ridge baseline（建议 1% 左右抽样）
# # =========================
# SAMPLE_TRAIN_RIDGE = 1
# SAMPLE_VALID_RIDGE = 1
# SAMPLE_TEST_RIDGE  = 1

# def run_ridge(y_mode: str):
#     X_tr, ytr_log, ytr_raw = load_sample_numpy(TRAIN_MONTHS, FEAT_COLS_RIDGE, cost_clip, time_clip, y_mode, SAMPLE_TRAIN_RIDGE)
#     X_va, yva_log, yva_raw = load_sample_numpy(VALID_MONTHS, FEAT_COLS_RIDGE, cost_clip, time_clip, y_mode, SAMPLE_VALID_RIDGE)
#     X_te, yte_log, yte_raw = load_sample_numpy(TEST_MONTHS,  FEAT_COLS_RIDGE, cost_clip, time_clip, y_mode, SAMPLE_TEST_RIDGE)

#     # Ridge: 标准化 + L2 正则
#     model = Pipeline([
#         ("imputer", SimpleImputer(strategy="median")),  # ✅ 关键：补 NaN
#         ("scaler", StandardScaler(with_mean=True, with_std=True)),
#         ("ridge", Ridge(alpha=5.0, random_state=42))
#     ])

#     model.fit(X_tr, ytr_log)

#     pred_va_log = model.predict(X_va)
#     pred_te_log = model.predict(X_te)

#     pred_va = np.expm1(pred_va_log)
#     pred_te = np.expm1(pred_te_log)

#     rmse_va, mae_va = eval_rmse_mae(pred_va, yva_raw)
#     rmse_te, mae_te = eval_rmse_mae(pred_te, yte_raw)

#     print(f"\n=== Ridge ({y_mode}) ===")
#     print(f"Valid RMSE={rmse_va:.4f}, MAE={mae_va:.4f}")
#     print(f"Test  RMSE={rmse_te:.4f}, MAE={mae_te:.4f}")

#     # 保存（可选）
#     import joblib
#     out = os.path.join(MODEL_DIR, f"ridge_{y_mode}_log.joblib")
#     joblib.dump(model, out)
#     print("[SAVED]", out)

#     del X_tr, ytr_log, ytr_raw, X_va, yva_log, yva_raw, X_te, yte_log, yte_raw
#     gc.collect()

# # =========================
# # 3) RandomForest baseline（一定要更小样本，例如 0.1%~0.3%）
# # =========================
# SAMPLE_TRAIN_RF = 0.2   # 0.2%（按你数据规模大概几十万级别）
# SAMPLE_VALID_RF = 0.2
# SAMPLE_TEST_RF  = 0.2

# def run_rf(y_mode: str):
#     # sample_pct 允许 float？我们这里用 mod(100) < sample_pct，需要 int。
#     # 所以 RF 用“千分位抽样”做法：mod(1000) < k
#     def load_rf(months, feat_cols, cost_clip, time_clip, y_mode: str, k: int):
#         X_list, ylog_list, yraw_list = [], [], []
#         for m in months:
#             path = month_path(m)

#             # ✅ 不重复 route_id（feat_cols 里已有）
#             lf = pl.scan_parquet(path).select(feat_cols + [Y_COST_COL, Y_TIME_COL])

#             lf = (
#                 lf.filter((pl.col(Y_COST_COL) >= 0) & (pl.col(Y_TIME_COL) > 0))
#                   .filter(pl.col("route_id").hash(seed=42).mod(1000) < k)  # 千分位抽样
#                   .with_columns([
#                       pl.col(Y_COST_COL).clip(cost_clip[0], cost_clip[1]).log1p().alias("y_cost_log"),
#                       pl.col(Y_TIME_COL).clip(time_clip[0], time_clip[1]).log1p().alias("y_time_log"),
#                   ])
#                   .select(feat_cols + ["y_cost_log","y_time_log", Y_COST_COL, Y_TIME_COL])
#             )

#             df = collect_streaming(lf)

#             # dtype 统一
#             for c in INT_COLS:
#                 if c in df.columns: df = df.with_columns(pl.col(c).cast(pl.Int32))
#             for c in BIN_COLS:
#                 if c in df.columns: df = df.with_columns(pl.col(c).cast(pl.Int8))
#             for c in FLOAT_COLS:
#                 if c in df.columns: df = df.with_columns(pl.col(c).cast(pl.Float32))

#             X = df.select(feat_cols).to_numpy()
#             if y_mode == "cost":
#                 y_log = df["y_cost_log"].to_numpy(); y_raw = df[Y_COST_COL].to_numpy()
#             else:
#                 y_log = df["y_time_log"].to_numpy(); y_raw = df[Y_TIME_COL].to_numpy()

#             # ✅ RF 不吃 NaN：这里直接补掉
#             X = np.nan_to_num(X, nan=0.0)

#             X_list.append(X); ylog_list.append(y_log); yraw_list.append(y_raw)
#             print(f"[OK] RF sample month={m:02d}, rows={X.shape[0]:,}")

#             del df
#             gc.collect()

#         return np.concatenate(X_list), np.concatenate(ylog_list), np.concatenate(yraw_list)

#     k = int(round(SAMPLE_TRAIN_RF * 1000))  # 0.2% -> k=2 (因为 mod(1000)<2)
#     k = max(1, min(50, k))                # 限制一下，避免写错
#     X_tr, ytr_log, ytr_raw = load_rf(TRAIN_MONTHS, FEAT_COLS_FULL, cost_clip, time_clip, y_mode, k)
#     X_va, yva_log, yva_raw = load_rf(VALID_MONTHS, FEAT_COLS_FULL, cost_clip, time_clip, y_mode, k)
#     X_te, yte_log, yte_raw = load_rf(TEST_MONTHS,  FEAT_COLS_FULL, cost_clip, time_clip, y_mode, k)

#     rf = RandomForestRegressor(
#         n_estimators=300,
#         max_depth=24,
#         min_samples_leaf=20,
#         max_features="sqrt",
#         n_jobs=-1,
#         random_state=42,
#         verbose=1
#     )
#     rf.fit(X_tr, ytr_log)

#     pred_va = np.expm1(rf.predict(X_va))
#     pred_te = np.expm1(rf.predict(X_te))

#     rmse_va, mae_va = eval_rmse_mae(pred_va, yva_raw)
#     rmse_te, mae_te = eval_rmse_mae(pred_te, yte_raw)

#     print(f"\n=== RandomForest ({y_mode}) ===")
#     print(f"Valid RMSE={rmse_va:.4f}, MAE={mae_va:.4f}")
#     print(f"Test  RMSE={rmse_te:.4f}, MAE={mae_te:.4f}")

#     import joblib
#     out = os.path.join(MODEL_DIR, f"rf_{y_mode}_log.joblib")
#     joblib.dump(rf, out)
#     print("[SAVED]", out)

#     del X_tr, ytr_log, ytr_raw, X_va, yva_log, yva_raw, X_te, yte_log, yte_raw
#     gc.collect()

# # ========= 跑两个目标 =========
# run_ridge("cost")
# run_ridge("time")

# # run_rf("cost")
# # run_rf("time")
import os
import gc
import numpy as np
import polars as pl
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

FEATURE_DIR = "/data/tlc_hvfhv_2024_features_wxhol"
MODEL_DIR   = "/data/tlc_hvfhv_2024_models"
os.makedirs(MODEL_DIR, exist_ok=True)

TRAIN_MONTHS = list(range(1, 10))   # 1-9
VALID_MONTHS = [10]
TEST_MONTHS  = [11, 12]

Y_COST_COL = "y_cost_no_tips"
Y_TIME_COL = "y_time"

BASE_FEAT_COLS = [
    "hour", "dow", "is_weekend", "month", "is_peak",
    "PULocationID", "DOLocationID", "same_zone", "route_id",
    "trip_miles", "log1p_trip_miles",
    "shared_request_flag", "wav_request_flag", "access_a_ride_flag",
    "originating_base_missing",
    "hvfhs_code", "dispatch_code", "origin_code",
]

EXTRA_FEAT_COLS = [
    "temperature_2m", "precipitation", "wind_speed_10m", "weather_code",
    "is_precip", "weather_missing",
    "is_holiday", "is_before_holiday", "is_after_holiday",
    "is_zero_miles",
]

FEAT_COLS = BASE_FEAT_COLS + EXTRA_FEAT_COLS

INT_COLS = ["PULocationID", "DOLocationID", "route_id", "hvfhs_code", "dispatch_code", "origin_code", "weather_code",
            "hour", "dow", "month"]
BIN_COLS = ["same_zone","is_weekend","is_peak",
            "shared_request_flag","wav_request_flag","access_a_ride_flag","originating_base_missing",
            "is_precip","weather_missing","is_holiday","is_before_holiday","is_after_holiday",
           "is_zero_miles"]
FLOAT_COLS = ["trip_miles","log1p_trip_miles","temperature_2m","precipitation","wind_speed_10m"]

def month_path(m: int) -> str:
    return os.path.join(FEATURE_DIR, f"features_2024-{m:02d}.parquet")

def collect_streaming(lf: pl.LazyFrame) -> pl.DataFrame:
    try:
        return lf.collect(engine="streaming")
    except TypeError:
        return lf.collect()

def estimate_clip_bounds(months, y_col: str, sample_pct: int = 2, q_low: float = 0.005, q_high: float = 0.995):
    paths = [month_path(m) for m in months]
    lf = pl.scan_parquet(paths).select(["route_id", y_col]).drop_nulls()
    lf_s = lf.filter(pl.col("route_id").hash(seed=42).mod(100) < sample_pct)
    df_s = collect_streaming(lf_s)

    y = df_s.get_column(y_col).to_numpy()
    y = y[np.isfinite(y)]
    if len(y) == 0:
        raise RuntimeError(f"Sample for {y_col} is empty. Check columns/data.")
    lo = float(np.quantile(y, q_low))
    hi = float(np.quantile(y, q_high))
    return lo, hi, len(y)

def load_months_sample_numpy(months, cost_clip, time_clip, y_mode: str, sample_pct: int = 2):
    assert y_mode in ("cost", "time")
    X_list, ylog_list, yraw_list = [], [], []

    for m in months:
        path = month_path(m)

        lf = pl.scan_parquet(path).select(FEAT_COLS + [Y_COST_COL, Y_TIME_COL])
        lf = (
            lf.filter((pl.col(Y_COST_COL) >= 0) & (pl.col(Y_TIME_COL) > 0))
              .filter(pl.col("route_id").hash(seed=42).mod(100) < sample_pct)
              .with_columns([
                  pl.col(Y_COST_COL).clip(cost_clip[0], cost_clip[1]).log1p().alias("y_cost_log"),
                  pl.col(Y_TIME_COL).clip(time_clip[0], time_clip[1]).log1p().alias("y_time_log"),
              ])
              .select(FEAT_COLS + ["y_cost_log","y_time_log", Y_COST_COL, Y_TIME_COL])
        )

        df = collect_streaming(lf)

        for c in INT_COLS:
            if c in df.columns:
                df = df.with_columns(pl.col(c).cast(pl.Int32))
        for c in BIN_COLS:
            if c in df.columns:
                df = df.with_columns(pl.col(c).cast(pl.Int8))
        for c in FLOAT_COLS:
            if c in df.columns:
                df = df.with_columns(pl.col(c).cast(pl.Float32))

        # 全部转 float32，避免 object
        df = df.with_columns([pl.col(c).cast(pl.Float32) for c in FEAT_COLS])

        X = df.select(FEAT_COLS).to_numpy()
        if y_mode == "cost":
            y_log = df["y_cost_log"].to_numpy().astype(np.float32, copy=False)
            y_raw = df[Y_COST_COL].to_numpy().astype(np.float32, copy=False)
        else:
            y_log = df["y_time_log"].to_numpy().astype(np.float32, copy=False)
            y_raw = df[Y_TIME_COL].to_numpy().astype(np.float32, copy=False)

        # Ridge 前统一把 NaN 留给 imputer（也可以这里填 0）
        X = X.astype(np.float32, copy=False)

        X_list.append(X); ylog_list.append(y_log); yraw_list.append(y_raw)
        print(f"[OK] sample month={m:02d}, rows={X.shape[0]:,}")

        del df
        gc.collect()

    X_all = np.concatenate(X_list, axis=0)
    ylog_all = np.concatenate(ylog_list, axis=0)
    yraw_all = np.concatenate(yraw_list, axis=0)

    del X_list, ylog_list, yraw_list
    gc.collect()
    return X_all, ylog_all, yraw_all

def rmse_mae(pred, y_true):
    pred = pred.astype(float)
    y_true = y_true.astype(float)
    rmse = float(np.sqrt(np.mean((pred - y_true) ** 2)))
    mae  = float(np.mean(np.abs(pred - y_true)))
    return rmse, mae

def run_ridge(y_mode: str, sample_train=2, sample_valid=2, sample_test=2, alpha=5.0):
    # clip bounds
    cost_lo, cost_hi, _ = estimate_clip_bounds(TRAIN_MONTHS, Y_COST_COL, sample_pct=2)
    time_lo, time_hi, _ = estimate_clip_bounds(TRAIN_MONTHS, Y_TIME_COL, sample_pct=2)
    cost_clip = (cost_lo, cost_hi)
    time_clip = (time_lo, time_hi)

    # load
    X_tr, ytr_log, ytr_raw = load_months_sample_numpy(TRAIN_MONTHS, cost_clip, time_clip, y_mode, sample_pct=sample_train)
    X_va, yva_log, yva_raw = load_months_sample_numpy(VALID_MONTHS, cost_clip, time_clip, y_mode, sample_pct=sample_valid)
    X_te, yte_log, yte_raw = load_months_sample_numpy(TEST_MONTHS,  cost_clip, time_clip, y_mode, sample_pct=sample_test)

    print(f"[rows] Ridge {y_mode}: train={len(ytr_log):,}, valid={len(yva_log):,}, test={len(yte_log):,}")

    # Ridge baseline：impute -> standardize -> ridge
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("ridge", Ridge(alpha=alpha, random_state=42)),
    ])

    model.fit(X_tr, ytr_log)

    pred_va_raw = np.expm1(model.predict(X_va))
    pred_te_raw = np.expm1(model.predict(X_te))

    rmse_va, mae_va = rmse_mae(pred_va_raw, yva_raw)
    rmse_te, mae_te = rmse_mae(pred_te_raw, yte_raw)

    print(f"\n=== Ridge ({y_mode}) raw-scale eval ===")
    print(f"Valid RMSE={rmse_va:.4f}, MAE={mae_va:.4f}")
    print(f"Test  RMSE={rmse_te:.4f}, MAE={mae_te:.4f}")

    out = os.path.join(MODEL_DIR, f"ridge_{y_mode}_log_wxhol_alpha{alpha}.joblib")
    joblib.dump(model, out)
    print("[SAVED]", out)

    del X_tr, ytr_log, ytr_raw, X_va, yva_log, yva_raw, X_te, yte_log, yte_raw
    gc.collect()

# 默认同口径：train/valid/test 都 2%
run_ridge("cost", sample_train=2, sample_valid=2, sample_test=2, alpha=5.0)
run_ridge("time", sample_train=2, sample_valid=2, sample_test=2, alpha=5.0)


[OK] sample month=01, rows=382,657
[OK] sample month=02, rows=376,097
[OK] sample month=03, rows=410,826
[OK] sample month=04, rows=383,504
[OK] sample month=05, rows=397,563
[OK] sample month=06, rows=387,167
[OK] sample month=07, rows=370,993
[OK] sample month=08, rows=373,504
[OK] sample month=09, rows=368,644
[OK] sample month=10, rows=382,503
[OK] sample month=11, rows=387,473
[OK] sample month=12, rows=407,435
[rows] Ridge cost: train=3,450,955, valid=382,503, test=794,908

=== Ridge (cost) raw-scale eval ===
Valid RMSE=11.8278, MAE=6.9670
Test  RMSE=13.3622, MAE=7.5705
[SAVED] /data/tlc_hvfhv_2024_models/ridge_cost_log_wxhol_alpha5.0.joblib
[OK] sample month=01, rows=382,657
[OK] sample month=02, rows=376,097
[OK] sample month=03, rows=410,826
[OK] sample month=04, rows=383,504
[OK] sample month=05, rows=397,563
[OK] sample month=06, rows=387,167
[OK] sample month=07, rows=370,993
[OK] sample month=08, rows=373,504
[OK] sample month=09, rows=368,644
[OK] sample month=10, rows=38

Ridge 可作为线性基线，在大样本下仍能高效训练（百万级样本可行），但由于打车费用/时长与时空特征之间存在显著的非线性与交互效应，线性模型难以刻画复杂模式，因此性能显著弱于基于梯度提升树的 XGBoost / LightGBM。